# 📘 Notebook 02 — Chain-of-Thought Generation via DeepSeek-R1

**Goals:**
1. Connect to DeepSeek-R1 (teacher model)
2. Generate CoT reasoning traces for 500 ILDC cases
3. Save incrementally to `cot_data.jsonl`
4. Format CoT data for SFT training

**Runtime:** ~1 hr (API calls) | **GPU:** Not required | **Cost:** ~$1-2

## 1. Setup

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DEEPSEEK_API_KEY, COT_NUM_SAMPLES, COT_DATA_PATH
from src.data_utils import load_legal_datasets
from src.cot_generator import create_deepseek_client, generate_cot_batch

print(f"DeepSeek API key: {'✅ Found' if DEEPSEEK_API_KEY else '❌ Missing'}")
print(f"Samples to generate: {COT_NUM_SAMPLES}")
print(f"Output path: {COT_DATA_PATH}")

## 2. Load ILDC Dataset

In [ ]:
ildc, _ = load_legal_datasets()
print(f"\nILDC has {len(ildc)} cases. We'll process the first {COT_NUM_SAMPLES}.")

## 3. Test Single CoT Generation

In [ ]:
from src.cot_generator import create_deepseek_client, get_cot_from_teacher

client = create_deepseek_client()

# Test with first case
test_facts = ildc[0].get('facts', ildc[0].get('text', ''))[:1000]
result = get_cot_from_teacher(client, test_facts)

print("=== REASONING TRACE (first 500 chars) ===")
print(result['reasoning'][:500])
print("\n=== ANSWER (first 500 chars) ===")
print(result['answer'][:500])

## 4. Generate Full Batch (500 Cases)

> **⚠️ This takes ~1 hour and costs ~$1-2.** 
> Data is saved every 50 samples to protect against crashes.

In [ ]:
cot_data = generate_cot_batch(
    ildc_dataset=ildc,
    client=client,
    n_samples=COT_NUM_SAMPLES,
    save_path=COT_DATA_PATH,
    save_every=50,
)

print(f"\n🎉 Generated {len(cot_data)} CoT samples!")

## 5. Inspect Saved Data

In [ ]:
import json

with open(COT_DATA_PATH, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"Total saved samples: {len(lines)}")

# Show a random sample
sample = json.loads(lines[42])
print(f"\n=== Sample #42 ===")
print(f"Facts (first 200 chars): {sample['facts'][:200]}")
print(f"Reasoning length: {len(sample['reasoning'])} chars")
print(f"Answer length: {len(sample['answer'])} chars")

## 6. Format CoT for Training

In [ ]:
from src.data_utils import load_cot_dataset

cot_dataset = load_cot_dataset(COT_DATA_PATH)

# Preview formatted sample
print("=== Formatted CoT Sample ===")
print(cot_dataset[0]['text'][:800])

## 7. Summary

✅ **Completed:**
- Connected to DeepSeek-R1 teacher model
- Generated CoT reasoning traces for 500 cases
- Saved to `cot_data.jsonl` with incremental checkpoints
- Formatted CoT data for SFT training

**Next → Notebook 03:** SFT fine-tuning with LoRA